# nb_gold_open_orders (Python notebook)

Example of a **Python notebook** (no Spark) that uses `fabric_utils` in python mode:
an hourly Gold load of the orders that haven't shipped yet, for an operations screen.

- **Kernel:** Python 3.12 (Fabric Python notebook), default size (2 vCores, 16 GB).
- **Library:** Python notebooks can't use an Environment. The first code cell installs the
  `fabric_enterprise_utils` wheel that `deploy.py` leaves in the folder of the variable
  `fabric_utils_libs` (Variable Library `vl_environment`), with `pip install --no-deps`.
- **Reads:** Silver `orders`, `customers`. **Writes:** Gold `open_orders` (overwrite) and one row in
  Control `notebook_logs`.
- **Permissions:** the identity running it needs read on the control lakehouse (the wheel) and on Silver,
  write on Gold and Control.


In [ ]:
# Parameters (a pipeline can override them)
notebook_name = "nb_gold_open_orders"


### Install fabric_utils
Same cell in every Python notebook. Skipped if the session already has the library.
`pip` runs as a Python process, not `%pip`: inline `%pip install` is disabled by default in pipeline runs.

In [ ]:
import importlib.util, subprocess, sys
import notebookutils

if importlib.util.find_spec("fabric_utils") is None:
    libs = notebookutils.variableLibrary.getLibrary("vl_environment").fabric_utils_libs
    whl = max((f for f in notebookutils.fs.ls(libs) if f.name.endswith(".whl")), key=lambda f: f.name)
    notebookutils.fs.cp(whl.path, f"file:/tmp/{whl.name}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-q", f"/tmp/{whl.name}"], check=True)


In [ ]:
from fabric_utils import *                       # python mode: no Spark in this session
import fabric_utils
import pyarrow as pa

print(f"fabric_utils {fabric_utils.__version__}, mode {FABRIC_UTILS_ENGINE}")

paths = get_paths()                              # dev or pro, from the Variable Library
log = NotebookLogger(notebook_name, layer="gold")


### Destination schema
The pyarrow equivalent of the `StructType` in a Spark notebook: column order and types of the Gold table.

In [ ]:
destination_schema = pa.schema([
    pa.field("order_id", pa.string()),
    pa.field("customer_id", pa.string()),
    pa.field("customer_name", pa.string()),
    pa.field("promised_date", pa.date32()),
    pa.field("status", pa.string()),
    pa.field("days_late", pa.int32()),
    pa.field("load_date", pa.timestamp("us", tz="UTC")),
])


### Transform
One DuckDB view per Delta table; DuckDB only reads the columns and rows the query uses.

In [ ]:
try:
    con = duckdb_connect()                       # UTC session + OneLake token
    register_delta_views(con, paths["path_silver"], ["orders", "customers"])

    rel = con.sql("""
        SELECT
            o.order_id,
            o.customer_id,
            c.customer_name,
            CAST(o.promised_date AS DATE)                                   AS promised_date,
            o.status,
            CAST(greatest(date_diff('day', CAST(o.promised_date AS DATE), current_date), 0) AS INTEGER) AS days_late,
            now()                                                           AS load_date
        FROM orders o
        LEFT JOIN customers c USING (customer_id)
        WHERE o.status <> 'SHIPPED'
    """)

    save_table_arrow(rel, paths["path_gold"], "open_orders", schema=destination_schema)
    log.end()
except Exception as e:
    log.log_error(e)
    raise
